> [!WARNING]
> **OFFLINE SUBMISSION NOTEBOOK**
> Internet is disabled. Upload the following as Kaggle datasets:
> 1. `b3-pretrained` — containing `dinov2_base_patch14.pth` and `convnext_base_in22k_384.pth`


# B3: DINOv2 + ConvNeXt Ensemble (Zero-Shot)

**Author:** Mridankan Mandal

**Competition:** [CSIRO Image2Biomass](https://www.kaggle.com/competitions/csiro-biomass)

---

## Overview

Zero-shot ensemble inference using pretrained DINOv2-Base and ConvNeXt-Base backbones.
Models are loaded directly via `timm` with `pretrained=True` (auto-downloads from HuggingFace).
No fine-tuning is performed.

- **DINOv2-Base:** `vit_base_patch14_dinov2.lvd142m`
- **ConvNeXt-Base:** `convnext_base.fb_in22k_ft_in1k_384`
- **Ensemble:** 0.55 × DINOv2 + 0.45 × ConvNeXt


In [ ]:
# --- Setup: configure local pretrained weight paths ---
import os
import glob as _glob

def _find(slug, pattern=None):
    """Find Kaggle dataset dir. If pattern given, searches subdirs too."""
    for base in [f'/kaggle/input/{slug}', *_glob.glob(f'/kaggle/input/datasets/*/{slug}')]:
        if not os.path.isdir(base): continue
        if pattern:
            if _glob.glob(os.path.join(base, pattern)): return base
            for sub in _glob.glob(os.path.join(base, '*')):
                if os.path.isdir(sub) and _glob.glob(os.path.join(sub, pattern)):
                    return sub
        return base
    raise FileNotFoundError(f"Dataset '{slug}' not found in /kaggle/input/")

WEIGHT_DIR = _find('b3-pretrained', '*.pth')
WEIGHT_MAP = {
    'vit_base_patch14_dinov2.lvd142m': 'dinov2_base_patch14.pth',
    'convnext_base.fb_in22k_ft_in1k_384': 'convnext_base_in22k_384.pth',
}
for k, v in WEIGHT_MAP.items():
    p = os.path.join(WEIGHT_DIR, v)
    print(f"  {k}: {'OK' if os.path.exists(p) else 'MISSING'} ({p})")


In [ ]:
import os, gc, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from PIL import Image
from tqdm.auto import tqdm
import timm
from timm.data import resolve_model_data_config
from torchvision import transforms

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR = '/kaggle/input/competitions/csiro-biomass'
TEST_CSV_PATH = os.path.join(DATA_DIR, 'test.csv')
TEST_IMG_DIR = os.path.join(DATA_DIR, 'test')
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, 'sample_submission.csv')

BATCH_SIZE = 1
NUM_TARGETS = 5
TARGET_NAMES = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']

DINO_MODEL_NAME = 'vit_base_patch14_dinov2.lvd142m'
CONV_MODEL_NAME = 'convnext_base.fb_in22k_ft_in1k_384'
W_DINO, W_CONV = 0.55, 0.45

print(f'Device: {DEVICE}')


In [ ]:
# Load test data
test_df = pd.read_csv(TEST_CSV_PATH)
img_ids = test_df['image_path'].apply(lambda x: os.path.basename(x).replace('.jpg', '')).unique()
print(f'Found {len(img_ids)} unique test images')


In [ ]:
def get_preds(model_name, model_label):
    """Run inference using a timm pretrained model."""
    print(f'Running: {model_label} ({model_name})')
    model = timm.create_model(model_name, pretrained=False, num_classes=NUM_TARGETS)
    # Load pretrained backbone weights from local dataset
    wt_path = os.path.join(WEIGHT_DIR, WEIGHT_MAP[model_name])
    if os.path.exists(wt_path):
        sd = torch.load(wt_path, map_location='cpu', weights_only=True)
        model.load_state_dict(sd, strict=False)
        print(f'  Loaded backbone weights from {wt_path}')
    model = model.to(DEVICE).eval()

    data_cfg = resolve_model_data_config(model)
    img_size = data_cfg['input_size'][-1]
    print(f'  Input size: {img_size}x{img_size}')

    val_tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    preds_accum = np.zeros((len(img_ids), NUM_TARGETS), dtype=np.float32)
    with torch.no_grad():
        for idx, img_id in enumerate(tqdm(img_ids, desc=f'{model_label}')):
            img_path = os.path.join(TEST_IMG_DIR, f'{img_id}.jpg')
            if not os.path.exists(img_path):
                continue
            img = Image.open(img_path).convert('RGB')
            img_t = val_tfm(img).unsqueeze(0).to(DEVICE)
            with autocast(device_type=DEVICE):
                out = model(img_t)
            preds_accum[idx] = out.cpu().numpy()[0]
    print(f'  Shape: {preds_accum.shape}')

    del model; gc.collect(); torch.cuda.empty_cache()
    return preds_accum

preds_dino = get_preds(DINO_MODEL_NAME, 'DINOv2')
preds_conv = get_preds(CONV_MODEL_NAME, 'ConvNeXt')


In [ ]:
# Weighted ensemble
print(f'Ensembling: {W_DINO}*DINOv2 + {W_CONV}*ConvNeXt')
preds_final = W_DINO * preds_dino + W_CONV * preds_conv

test_csv = pd.read_csv(TEST_CSV_PATH)
pred_map = {}
for i, img_id in enumerate(img_ids):
    for j, target_name in enumerate(TARGET_NAMES):
        pred_map[(img_id, target_name)] = float(preds_final[i, j])

test_csv['image_id'] = test_csv['sample_id'].str.split('__').str[0]
test_csv['target'] = test_csv.apply(
    lambda row: pred_map.get((row['image_id'], row['target_name']), 0.0), axis=1
)
df_sub = test_csv[['sample_id', 'target']].copy()

if os.path.exists(SAMPLE_SUB_PATH):
    sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
    df_sub = sample_sub[['sample_id']].merge(df_sub, on='sample_id', how='left')
    df_sub['target'] = df_sub['target'].fillna(0.0)

df_sub.to_csv('submission.csv', index=False)
print(f'Saved submission.csv, shape: {df_sub.shape}')
print(df_sub.head(10).to_string(index=False))
